# 🔍 CRACKWATCH — YOLOv8 Road Damage Detection Training

This notebook trains a YOLOv8n model on the RDD2022 road damage dataset.

**Classes:** D00 (Longitudinal Crack), D10 (Transverse Crack), D20 (Alligator Crack), D40 (Pothole)

**Steps:**
1. Install dependencies
2. Download dataset from Roboflow
3. Train YOLOv8n for 50 epochs
4. Download `best.pt` to your laptop

⚡ **Runtime → Change runtime type → T4 GPU** before running!

In [ ]:
# Step 1: Install dependencies
!pip install ultralytics roboflow -q
print('✅ Dependencies installed')

In [ ]:
# Step 2: Download RDD2022 dataset from Roboflow
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("rdd2022").project("test-dataset-u1qbi")
version = project.version(1)
dataset = version.download("yolov8")
print(f'✅ Dataset downloaded to: {dataset.location}')

In [ ]:
# Step 2b (ALTERNATIVE): If Roboflow API key doesn't work,
# use this direct download method instead:
#
# !pip install gdown -q
# import gdown
# # Download RDD2022 from the official repository
# !git clone https://github.com/sekilab/RoadDamageDetector.git
# print('✅ RDD2022 downloaded via GitHub')
#
# Then you'll need to convert to YOLO format — uncomment if needed

In [ ]:
# Step 3: Train YOLOv8n
from ultralytics import YOLO

# Load YOLOv8 nano (fastest, best for hackathon/real-time)
model = YOLO('yolov8n.pt')

# Train on the road damage dataset
results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='crackwatch',
    patience=10,        # Early stopping
    save=True,
    save_period=10,     # Save checkpoint every 10 epochs
    plots=True,
    verbose=True,
)

print('\n✅ Training complete!')
print(f'Best model saved at: runs/detect/crackwatch/weights/best.pt')

In [ ]:
# Step 4: Evaluate the model
model = YOLO('runs/detect/crackwatch/weights/best.pt')
metrics = model.val()

print(f'\n📊 Results:')
print(f'  mAP50:    {metrics.box.map50:.3f}')
print(f'  mAP50-95: {metrics.box.map:.3f}')
print(f'  Precision: {metrics.box.mp:.3f}')
print(f'  Recall:    {metrics.box.mr:.3f}')

In [ ]:
# Step 5: Test on a sample image
import glob
from IPython.display import Image, display

# Pick a test image
test_images = glob.glob(f'{dataset.location}/test/images/*.jpg')[:3]

for img_path in test_images:
    results = model(img_path)
    for r in results:
        annotated = r.plot()
        # Save and display
        import cv2
        out_path = img_path.replace('.jpg', '_detected.jpg')
        cv2.imwrite(out_path, annotated)
        display(Image(filename=out_path, width=600))
        print(f'Detections: {len(r.boxes)}')
        print('---')

In [ ]:
# Step 6: Download the trained model
# This is the file you need for CRACKWATCH backend
from google.colab import files

# Download best.pt — put this in crackwatch/backend/model/best.pt
files.download('runs/detect/crackwatch/weights/best.pt')
print('\n🎯 Download best.pt and place it in: crackwatch/backend/model/best.pt')
print('The backend will auto-detect and use it — no code changes needed!')

In [ ]:
# BONUS: Also export to ONNX for faster CPU inference (optional)
# model.export(format='onnx')
# files.download('runs/detect/crackwatch/weights/best.onnx')